# Postgres pipelines: file -> Postgres -> files -> Postgres

The Spark jobs talk to Postgres over JDBC (the driver jar is in the Spark image). From the
notebook we check the results directly in Postgres with `lab.pg()`.

1. `pg-file-to-pg`: Parquet on RustFS -> `shop.orders`
2. `pg-pg-to-files`: Postgres -> Parquet / ORC / CSV, with parallel reads and pushdown
3. `pg-pg-to-pg`: Postgres -> Postgres, full refresh plus an idempotent upsert

In [ ]:
import kyuubi_lab as lab          # client/kyuubi_lab.py (on PYTHONPATH in the Jupyter image)
import pandas as pd
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200)
RID = lab.new_run_id()            # one id for every job in this notebook run
print("Kyuubi REST", lab.REST_URL, "| JDBC", f"{lab.JDBC_HOST}:{lab.JDBC_PORT}", "| user", lab.USER, "| run", RID)

In [ ]:
r = lab.run('pg-file-to-pg', run_id=RID)
lab.log_tail(r, 'rows')

In [ ]:
lab.pg("SELECT count(*) AS orders, round(sum(amount)::numeric, 2) AS revenue, max(loaded_at) AS loaded_at FROM orders")

In [ ]:
lab.pg("SELECT * FROM orders ORDER BY order_id LIMIT 5")

In [ ]:
r = lab.run('pg-pg-to-files', run_id=RID)
lab.s3_ls('data', 'output/pg_export/').head(20)

Read one of the exports back through Kyuubi SQL:

In [ ]:
lab.sql("SELECT * FROM parquet.`s3a://data/output/pg_export/customer_orders_parquet` LIMIT 5")

In [ ]:
r = lab.run('pg-pg-to-pg', run_id=RID)
lab.log_tail(r, 'upsert')

In [ ]:
lab.pg("SELECT * FROM daily_sales ORDER BY 1 LIMIT 10")

In [ ]:
lab.pg("SELECT * FROM customer_ltv ORDER BY revenue DESC LIMIT 10")

Run `pg-pg-to-pg` again: the upsert is idempotent, so `customer_ltv` keeps the same row count.

In [ ]:
before = int(lab.pg("SELECT count(*) n FROM customer_ltv")["n"][0])
lab.run('pg-pg-to-pg', run_id=RID)
after = int(lab.pg("SELECT count(*) n FROM customer_ltv")["n"][0])
print(before, after); assert before == after